# Classical baselines

Train word/character TF-IDF Logistic Regression and Linear SVM models using the training split and evaluate on validation.

In [1]:
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src.classical import train_classical_model, predict_classical_model, save_prediction_file
from src.evaluate import compute_metrics, append_result_row
from src.preprocess import PreprocessConfig, Preprocessor

DATA_DIR = PROJECT_ROOT / 'data'
train = pd.read_csv(DATA_DIR / 'train.tsv', sep='\t', header=None, names=['uid', 'text', 'label'])
val = pd.read_csv(DATA_DIR / 'validation.tsv', sep='\t', header=None, names=['uid', 'text', 'label'])
train['text'] = train['text'].fillna('').astype(str)
val['text'] = val['text'].fillna('').astype(str)

preprocessor = Preprocessor(PreprocessConfig()).fit(train['text'])
train_texts = [preprocessor.transform_text(text) for text in train['text']]
val_texts = [preprocessor.transform_text(text) for text in val['text']]
y_train = train['label'].astype(int)
y_val = val['label'].astype(int)

model, vectorizer = train_classical_model(train_texts, y_train, 'logistic_regression', 'word')
predictions, probabilities = predict_classical_model(model, vectorizer, val_texts)
results_dir = PROJECT_ROOT / 'results'
prediction_path = results_dir / 'predictions' / 'model.csv'
save_prediction_file(prediction_path, val['uid'], y_val, predictions, probabilities)
prediction_frame = pd.DataFrame({'uid': val['uid'], 'y_true': y_val, 'y_pred': predictions, 'p_neg': probabilities[:, 0], 'p_neu': probabilities[:, 1], 'p_pos': probabilities[:, 2]})
metrics = compute_metrics(prediction_frame)
append_result_row(results_dir / 'results.csv', {'model': 'logistic_regression', 'embedding': 'tfidf_word', 'normalized': True, 'accuracy': metrics['accuracy'], 'macro_f1': metrics['macro_f1'], 'weighted_f1': metrics['weighted_f1'], 'seed': 42})
preprocessor.save(PROJECT_ROOT / 'data' / 'processed' / 'preprocessor.json')
prediction_path, metrics, predictions[:10], probabilities[:2]


(array([2, 0, 0, 1, 2, 2, 2, 2, 2, 2]),
 array([[0.092259  , 0.26461742, 0.64312357],
        [0.61477765, 0.27078196, 0.11444039]]))